# ElementalTask-RML  
## Notebook 03 — Constraint Drift Flags

This notebook converts emergence-order and function-vector geometry into lightweight training-monitoring signals.

**Goal**
- detect capability emergence ahead/behind expected order,
- identify geometry drift,
- track recovery toward stable capability structure.

**Pipeline**

`checkpoint → emergence rank → FV geometry → constraint flags`

Emergence ≠ magic. Monitor constraints. 📐


## 1. Setup

This notebook is designed to run from either the repo root or `notebooks_rml/` in Colab/GitHub.

It expects outputs from Notebook 01 and Notebook 02 in:

```text
notebooks_rml/results/
notebooks_rml/figures/
```

If those files are missing, this notebook creates a small fallback dataset so the monitoring logic can still be inspected.


In [ ]:

from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "notebooks_rml").exists() or (p / "dataset").exists() or (p / ".git").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks_rml"
if not NOTEBOOK_DIR.exists() and Path.cwd().name == "notebooks_rml":
    NOTEBOOK_DIR = Path.cwd()

FIG_DIR = NOTEBOOK_DIR / "figures"
RESULTS_DIR = NOTEBOOK_DIR / "results"
DOCS_DIR = NOTEBOOK_DIR / "docs"
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("FIG_DIR:", FIG_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


## 2. Load prior notebook outputs

Notebook 03 builds on prior results when available:

- `01_rank_stability_by_checkpoint.csv`
- `01_constraint_scores.csv`
- `01_emergence_order_table.csv`
- `01_pairwise_constraints.csv`
- `02_fv_drift_scores.csv`

Column names can vary slightly, so the loader normalizes common alternatives.


In [ ]:

def read_csv_if_exists(path):
    path = Path(path)
    if path.exists():
        print(f"Loaded: {path}")
        return pd.read_csv(path)
    print(f"Missing: {path}")
    return None

rank_df = read_csv_if_exists(RESULTS_DIR / "01_rank_stability_by_checkpoint.csv")
cgcs_df = read_csv_if_exists(RESULTS_DIR / "01_constraint_scores.csv")
emergence_df = read_csv_if_exists(RESULTS_DIR / "01_emergence_order_table.csv")
pairwise_df = read_csv_if_exists(RESULTS_DIR / "01_pairwise_constraints.csv")
fv_drift_df = read_csv_if_exists(RESULTS_DIR / "02_fv_drift_scores.csv")

# Also support files placed directly in notebooks_rml/ during early work.
if rank_df is None:
    rank_df = read_csv_if_exists(NOTEBOOK_DIR / "01_rank_stability_by_checkpoint.csv")
if cgcs_df is None:
    cgcs_df = read_csv_if_exists(NOTEBOOK_DIR / "01_constraint_scores.csv")
if emergence_df is None:
    emergence_df = read_csv_if_exists(NOTEBOOK_DIR / "01_emergence_order_table.csv")
if pairwise_df is None:
    pairwise_df = read_csv_if_exists(NOTEBOOK_DIR / "01_pairwise_constraints.csv")
if fv_drift_df is None:
    fv_drift_df = read_csv_if_exists(NOTEBOOK_DIR / "02_fv_drift_scores.csv")


## 3. Fallback data

The fallback data mirrors the toy developmental pattern from Notebooks 01–02:

- early unstable checkpoint geometry,
- increasing rank stability,
- decreasing function-vector drift,
- eventual recovery toward stable ordering.


In [ ]:

if rank_df is None:
    rank_df = pd.DataFrame({
        "checkpoint_order": [1000, 5000, 10000, 20000, 50000, 100000],
        "rank_stability": [0.58, 0.93, 0.93, 0.99, 0.99, 1.00],
    })

if cgcs_df is None:
    cgcs_df = pd.DataFrame({
        "checkpoint_order": [100000],
        "valid_constraints": [6],
        "violations": [0],
        "cgcs": [1.0],
    })

if emergence_df is None:
    emergence_df = pd.DataFrame({
        "task": [
            "simple:copying",
            "simple:uppercase",
            "simple:first_letter",
            "math:arithmetic",
            "compositional:copy_then_uppercase",
            "compositional:first_letter_then_uppercase",
        ],
        "emergence_checkpoint": [5000, 10000, 20000, 20000, 100000, 100000],
        "task_type": ["atomic", "atomic", "atomic", "atomic", "composite", "composite"],
    })

if pairwise_df is None:
    pairwise_df = pd.DataFrame({
        "component_task": ["simple:copying", "simple:uppercase", "simple:first_letter", "simple:uppercase"],
        "composite_task": ["compositional:copy_then_uppercase", "compositional:copy_then_uppercase", "compositional:first_letter_then_uppercase", "compositional:first_letter_then_uppercase"],
        "component_checkpoint": [5000, 10000, 20000, 10000],
        "composite_checkpoint": [100000, 100000, 100000, 100000],
        "valid_order": [True, True, True, True],
    })

if fv_drift_df is None:
    fv_drift_df = pd.DataFrame({
        "checkpoint_order": [1000, 5000, 10000, 20000, 50000, 100000],
        "fv_drift": [0.165, 0.365, 0.343, 0.303, 0.263, 0.000],
        "fv_rank_stability": [0.77, -0.31, -0.49, 0.14, 0.42, 1.0],
        "cgcs_fv": [0.75, 0.50, 0.50, 0.00, 0.50, 1.0],
    })

for name, df in [("rank_df", rank_df), ("cgcs_df", cgcs_df), ("emergence_df", emergence_df), ("pairwise_df", pairwise_df), ("fv_drift_df", fv_drift_df)]:
    print("\n", name)
    display(df.head())


## 4. Unified monitoring table

Create one row per checkpoint with:

```text
checkpoint_order | rank_stability | cgcs | fv_drift | fv_rank_stability | cgcs_fv
```


In [ ]:

def normalize_checkpoint_col(df):
    if df is None or df.empty:
        return df
    df = df.copy()
    candidates = ["checkpoint_order", "checkpoint", "step", "global_step", "tokens", "ckpt"]
    found = None
    for c in candidates:
        if c in df.columns:
            found = c
            break
    if found is None:
        df["checkpoint_order"] = np.arange(len(df))
    elif found != "checkpoint_order":
        df = df.rename(columns={found: "checkpoint_order"})
    df["checkpoint_order"] = pd.to_numeric(df["checkpoint_order"], errors="coerce")
    return df


def rename_first_existing(df, alternatives, target):
    if df is None:
        return df
    df = df.copy()
    if target in df.columns:
        return df
    for c in alternatives:
        if c in df.columns:
            return df.rename(columns={c: target})
    return df

rank_df = normalize_checkpoint_col(rank_df)
cgcs_df = normalize_checkpoint_col(cgcs_df)
fv_drift_df = normalize_checkpoint_col(fv_drift_df)

rank_df = rename_first_existing(rank_df, ["spearman", "spearman_to_final", "rank_corr", "rank_correlation"], "rank_stability")
cgcs_df = rename_first_existing(cgcs_df, ["minimal_cgcs", "constraint_score", "score"], "cgcs")
fv_drift_df = rename_first_existing(fv_drift_df, ["drift", "mean_abs_drift", "fv_geometry_drift"], "fv_drift")
fv_drift_df = rename_first_existing(fv_drift_df, ["spearman_to_final", "rank_stability_fv", "fv_spearman"], "fv_rank_stability")
fv_drift_df = rename_first_existing(fv_drift_df, ["fv_constraint_score", "constraint_score_fv"], "cgcs_fv")

checkpoints = sorted(set(rank_df["checkpoint_order"].dropna().tolist()) | set(fv_drift_df["checkpoint_order"].dropna().tolist()))
monitor_df = pd.DataFrame({"checkpoint_order": checkpoints})

if "rank_stability" in rank_df.columns:
    monitor_df = monitor_df.merge(rank_df[["checkpoint_order", "rank_stability"]].drop_duplicates(), on="checkpoint_order", how="left")

if "fv_drift" in fv_drift_df.columns:
    keep = [c for c in ["checkpoint_order", "fv_drift", "fv_rank_stability", "cgcs_fv"] if c in fv_drift_df.columns]
    monitor_df = monitor_df.merge(fv_drift_df[keep].drop_duplicates(), on="checkpoint_order", how="left")

if "cgcs" in cgcs_df.columns:
    if len(cgcs_df) == 1:
        monitor_df["cgcs"] = float(cgcs_df["cgcs"].iloc[0])
    else:
        monitor_df = monitor_df.merge(cgcs_df[["checkpoint_order", "cgcs"]].drop_duplicates(), on="checkpoint_order", how="left")
else:
    monitor_df["cgcs"] = np.nan

monitor_df = monitor_df.sort_values("checkpoint_order").reset_index(drop=True)
for col in ["rank_stability", "fv_drift", "fv_rank_stability", "cgcs_fv", "cgcs"]:
    if col in monitor_df.columns:
        monitor_df[col] = pd.to_numeric(monitor_df[col], errors="coerce").interpolate(limit_direction="both")

monitor_df


## 5. Monitoring heuristics

Flags are intentionally conservative and transparent.

| Flag | Meaning |
|---|---|
| `unstable_order` | emergence rank correlation remains low |
| `constraint_violation` | CGCS indicates ordering violations |
| `high_fv_drift` | function-vector geometry differs strongly from final geometry |
| `fv_geometry_unstable` | FV pairwise geometry rank is unstable |
| `stable` | ordering and geometry are both stable enough |
| `recovering` | instability is improving but not yet stable |


In [ ]:

THRESHOLDS = {
    "rank_stability_low": 0.50,
    "rank_stability_stable": 0.90,
    "cgcs_low": 0.70,
    "cgcs_stable": 0.90,
    "fv_drift_high": 0.30,
    "fv_drift_stable": 0.15,
    "fv_rank_low": 0.20,
    "fv_rank_stable": 0.70,
    "cgcs_fv_low": 0.50,
    "cgcs_fv_stable": 0.75,
}


def classify_row(row, thresholds=THRESHOLDS):
    flags = []
    rank = row.get("rank_stability", np.nan)
    cgcs = row.get("cgcs", np.nan)
    fv_drift = row.get("fv_drift", np.nan)
    fv_rank = row.get("fv_rank_stability", np.nan)
    cgcs_fv = row.get("cgcs_fv", np.nan)

    if pd.notna(rank) and rank < thresholds["rank_stability_low"]:
        flags.append("unstable_order")
    if pd.notna(cgcs) and cgcs < thresholds["cgcs_low"]:
        flags.append("constraint_violation")
    if pd.notna(fv_drift) and fv_drift > thresholds["fv_drift_high"]:
        flags.append("high_fv_drift")
    if pd.notna(fv_rank) and fv_rank < thresholds["fv_rank_low"]:
        flags.append("fv_geometry_unstable")
    if pd.notna(cgcs_fv) and cgcs_fv < thresholds["cgcs_fv_low"]:
        flags.append("fv_constraint_violation")

    stable_conditions = []
    if pd.notna(rank): stable_conditions.append(rank >= thresholds["rank_stability_stable"])
    if pd.notna(cgcs): stable_conditions.append(cgcs >= thresholds["cgcs_stable"])
    if pd.notna(fv_drift): stable_conditions.append(fv_drift <= thresholds["fv_drift_stable"])
    if pd.notna(fv_rank): stable_conditions.append(fv_rank >= thresholds["fv_rank_stable"])
    if pd.notna(cgcs_fv): stable_conditions.append(cgcs_fv >= thresholds["cgcs_fv_stable"])

    if flags:
        return ";".join(flags)
    if stable_conditions and all(stable_conditions):
        return "stable"
    return "recovering"

monitor_df["monitor_flag"] = monitor_df.apply(classify_row, axis=1)
monitor_df


## 6. Ahead / behind schedule detection

For a component → composite constraint:

```text
component task should emerge before composite task
```

Define:

```text
delta = composite_emergence_checkpoint - component_emergence_checkpoint
```

| Delta | Flag |
|---|---|
| negative | ahead-of-order composite |
| small/positive | expected |
| very large | behind schedule / delayed composition |


In [ ]:

def normalize_pairwise_columns(df):
    df = df.copy()
    rename_map = {}
    alternatives = {
        "component_task": ["component", "prerequisite", "atomic_task", "source_task"],
        "composite_task": ["composite", "target_task", "complex_task"],
        "component_checkpoint": ["component_emergence_checkpoint", "component_ckpt", "source_checkpoint"],
        "composite_checkpoint": ["composite_emergence_checkpoint", "composite_ckpt", "target_checkpoint"],
    }
    for target, alts in alternatives.items():
        if target not in df.columns:
            for alt in alts:
                if alt in df.columns:
                    rename_map[alt] = target
                    break
    return df.rename(columns=rename_map)

pairwise_df = normalize_pairwise_columns(pairwise_df)

if pairwise_df is not None and emergence_df is not None:
    if "component_checkpoint" not in pairwise_df.columns or "composite_checkpoint" not in pairwise_df.columns:
        emergence_map = dict(zip(emergence_df["task"], emergence_df["emergence_checkpoint"]))
        if "component_task" in pairwise_df.columns:
            pairwise_df["component_checkpoint"] = pairwise_df["component_task"].map(emergence_map)
        if "composite_task" in pairwise_df.columns:
            pairwise_df["composite_checkpoint"] = pairwise_df["composite_task"].map(emergence_map)

pairwise_monitor = pairwise_df.copy()
pairwise_monitor["component_checkpoint"] = pd.to_numeric(pairwise_monitor["component_checkpoint"], errors="coerce")
pairwise_monitor["composite_checkpoint"] = pd.to_numeric(pairwise_monitor["composite_checkpoint"], errors="coerce")
pairwise_monitor["delta"] = pairwise_monitor["composite_checkpoint"] - pairwise_monitor["component_checkpoint"]

ckpt_span = max(1, monitor_df["checkpoint_order"].max() - monitor_df["checkpoint_order"].min())
behind_threshold = 0.50 * ckpt_span


def classify_schedule_delta(delta):
    if pd.isna(delta): return "unknown"
    if delta < 0: return "ahead_of_order"
    if delta > behind_threshold: return "behind_schedule"
    return "expected_order"

pairwise_monitor["schedule_flag"] = pairwise_monitor["delta"].apply(classify_schedule_delta)
pairwise_monitor


## 7. Monitoring timeline

This plot combines rank-order stability, FV rank stability, FV drift, and available CGCS references.


In [ ]:

plt.figure(figsize=(14, 7))
x = monitor_df["checkpoint_order"]

if "rank_stability" in monitor_df.columns:
    plt.plot(x, monitor_df["rank_stability"], marker="o", label="emergence rank stability")
if "fv_rank_stability" in monitor_df.columns:
    plt.plot(x, monitor_df["fv_rank_stability"], marker="o", label="FV geometry rank stability")
if "cgcs_fv" in monitor_df.columns:
    plt.plot(x, monitor_df["cgcs_fv"], marker="o", label="CGCS-FV")
if "cgcs" in monitor_df.columns:
    plt.plot(x, monitor_df["cgcs"], marker="o", linestyle="--", label="minimal CGCS")
if "fv_drift" in monitor_df.columns:
    max_drift = monitor_df["fv_drift"].max()
    if pd.notna(max_drift) and max_drift > 0:
        plt.plot(x, 1 - (monitor_df["fv_drift"] / max_drift), marker="o", label="FV drift recovery (inverted)")

plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(0.7, linestyle=":", linewidth=1)
plt.ylim(-1.05, 1.05)
plt.xlabel("Checkpoint order")
plt.ylabel("Monitoring score")
plt.title("ElementalTask-RML: Constraint Drift Monitoring Timeline")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()

path = FIG_DIR / "03_constraint_drift_flags.png"
plt.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 8. Geometry recovery dashboard

A compact dashboard view of recovery:

- rank stability rises,
- FV drift falls,
- FV constraint score rises.


In [ ]:

plt.figure(figsize=(14, 7))
x = monitor_df["checkpoint_order"]

if "fv_drift" in monitor_df.columns:
    drift = monitor_df["fv_drift"].astype(float)
    dmin, dmax = drift.min(), drift.max()
    if dmax > dmin:
        drift_recovery = 1 - ((drift - dmin) / (dmax - dmin))
    else:
        drift_recovery = pd.Series(np.ones(len(drift)), index=drift.index)
    plt.plot(x, drift_recovery, marker="o", label="FV drift recovery")

if "rank_stability" in monitor_df.columns:
    plt.plot(x, monitor_df["rank_stability"], marker="o", label="emergence rank stability")
if "cgcs_fv" in monitor_df.columns:
    plt.plot(x, monitor_df["cgcs_fv"], marker="o", label="FV constraint score")

for _, row in monitor_df.iterrows():
    label = str(row["monitor_flag"]).replace(";", "\n")
    plt.text(row["checkpoint_order"], 0.03, label, rotation=45, ha="right", va="bottom", fontsize=8)

plt.ylim(-0.05, 1.08)
plt.xlabel("Checkpoint order")
plt.ylabel("Recovered/stable score")
plt.title("ElementalTask-RML: Geometry Recovery Monitor")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()

path = FIG_DIR / "03_geometry_recovery.png"
plt.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 9. Ahead / behind schedule figure

Positive deltas mean components emerged before composites. Negative deltas would indicate an ahead-of-order composite capability.


In [ ]:

plot_df = pairwise_monitor.copy()
plot_df["constraint"] = plot_df["component_task"].astype(str) + " → " + plot_df["composite_task"].astype(str)
plot_df = plot_df.sort_values("delta")

plt.figure(figsize=(14, max(5, 0.55 * len(plot_df))))
plt.barh(plot_df["constraint"], plot_df["delta"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.axvline(behind_threshold, linestyle=":", linewidth=1)
plt.xlabel("Composite checkpoint - component checkpoint")
plt.ylabel("Component → composite constraint")
plt.title("ElementalTask-RML: Ahead / Behind Schedule Constraints")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()

path = FIG_DIR / "03_ahead_behind_schedule.png"
plt.savefig(path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 10. Export monitoring tables

Notebook 03 saves:

```text
03_monitoring_table.csv
03_pairwise_schedule_flags.csv
03_monitoring_summary.json
```


In [ ]:

monitor_path = RESULTS_DIR / "03_monitoring_table.csv"
pairwise_path = RESULTS_DIR / "03_pairwise_schedule_flags.csv"
summary_path = RESULTS_DIR / "03_monitoring_summary.json"

monitor_df.to_csv(monitor_path, index=False)
pairwise_monitor.to_csv(pairwise_path, index=False)

summary = {
    "n_checkpoints": int(len(monitor_df)),
    "n_pairwise_constraints": int(len(pairwise_monitor)),
    "monitor_flag_counts": monitor_df["monitor_flag"].value_counts().to_dict(),
    "schedule_flag_counts": pairwise_monitor["schedule_flag"].value_counts().to_dict(),
    "thresholds": THRESHOLDS,
    "behind_threshold": float(behind_threshold),
}
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", monitor_path)
print("Saved:", pairwise_path)
print("Saved:", summary_path)
summary


## 11. Markdown report export

This creates a small lab-report style summary suitable for `notebooks_rml/docs/` or direct README adaptation.


In [ ]:

report = f'''# Notebook 03 — Constraint Drift Flags

ElementalTask-RML converts emergence ordering and function-vector geometry into lightweight training-monitoring signals.

## Inputs

- Notebook 01 emergence-order outputs
- Notebook 02 function-vector drift outputs

## Monitoring pipeline

`checkpoint → emergence rank → FV geometry → constraint flags`

## Summary

- Checkpoints analyzed: {summary['n_checkpoints']}
- Pairwise component → composite constraints: {summary['n_pairwise_constraints']}

## Monitor flag counts

{pd.Series(summary['monitor_flag_counts']).to_markdown()}

## Schedule flag counts

{pd.Series(summary['schedule_flag_counts']).to_markdown()}

## Figures

![Constraint Drift Flags](../figures/03_constraint_drift_flags.png)

![Geometry Recovery](../figures/03_geometry_recovery.png)

![Ahead / Behind Schedule](../figures/03_ahead_behind_schedule.png)

## Interpretation

ElementalTask suggests emergence ordering and function-vector geometry become increasingly stable during training.

This notebook explores whether lightweight monitoring heuristics can detect unstable emergence, capability ordering violations, and geometry drift before training completion.

Emergence ≠ magic. Monitor constraints. 📐
'''

report_path = DOCS_DIR / '03_constraint_drift_flags.md'
report_path.write_text(report)
print('Saved:', report_path)
print(report[:1200])


## 12. Optional: Colab zip download

Uncomment this cell in Colab to download Notebook 03 outputs.


In [ ]:

# # OPTIONAL COLAB DOWNLOAD
# # Uncomment this cell in Google Colab to zip and download Notebook 03 outputs.
#
# import zipfile
# from google.colab import files
#
# EXPORT_NAME = "elementaltask_rml_notebook03_outputs.zip"
# export_paths = [
#     FIG_DIR / "03_constraint_drift_flags.png",
#     FIG_DIR / "03_geometry_recovery.png",
#     FIG_DIR / "03_ahead_behind_schedule.png",
#     RESULTS_DIR / "03_monitoring_table.csv",
#     RESULTS_DIR / "03_pairwise_schedule_flags.csv",
#     RESULTS_DIR / "03_monitoring_summary.json",
#     DOCS_DIR / "03_constraint_drift_flags.md",
# ]
#
# with zipfile.ZipFile(EXPORT_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for p in export_paths:
#         p = Path(p)
#         if p.exists():
#             zf.write(p, arcname=str(p.relative_to(NOTEBOOK_DIR)))
#
# files.download(EXPORT_NAME)


## 13. Next notebook direction

Notebook 04 can move from monitoring flags to forecasting:

```text
components at early checkpoints → predict composite emergence trajectory
```

Candidate title:

```text
04_compositional_forecasting.ipynb
```

Core question:

> Can component trajectories forecast composite capability emergence before training completes?
